In [ ]:
import os, random, glob, math, time
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from tqdm import tqdm
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

In [ ]:
ROOT = './confirmed_fronts'
DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
MAX_PER_CLASS = 500
IMG_SIZE = 224
BATCH_SIZE = 64
NUM_WORKERS = 16
SEED = 42

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True
print(f'Device: {DEVICE}')

In [ ]:
TARGET_COLORS = ['Black', 'Blue', 'Brown', 'Green', 'Orange', 'Red', 'White', 'Yellow']
NUM_CLASSES = len(TARGET_COLORS)
color2idx = {c: i for i, c in enumerate(TARGET_COLORS)}

all_paths, all_labels = [], []
for path in glob.iglob(f'{ROOT}/*/*/*.jpg'):
    color = path.split('/')[-1].split('$$')[3]
    if color in color2idx:
        all_paths.append(path)
        all_labels.append(color2idx[color])

print(f'Found {len(all_paths)} images')
for c, n in sorted(Counter(TARGET_COLORS[y] for y in all_labels).items()):
    print(f'  {c:8s} {n}')

In [ ]:
labels_arr = np.array(all_labels)
rng = random.Random(SEED)

balanced_idx = []
for c in range(NUM_CLASSES):
    ids = np.where(labels_arr == c)[0].tolist()
    n_take = min(MAX_PER_CLASS, len(ids))
    balanced_idx.extend(rng.sample(ids, n_take))

balanced_idx = np.array(balanced_idx)
balanced_y = labels_arr[balanced_idx]

train_idx, test_idx = train_test_split(balanced_idx, test_size=0.2, stratify=balanced_y, random_state=SEED)
train_idx, val_idx = train_test_split(train_idx, test_size=0.2, stratify=labels_arr[train_idx], random_state=SEED)

train_paths = [all_paths[i] for i in train_idx]
train_labels = [all_labels[i] for i in train_idx]
val_paths   = [all_paths[i] for i in val_idx]
val_labels  = [all_labels[i] for i in val_idx]
test_paths  = [all_paths[i] for i in test_idx]
test_labels = [all_labels[i] for i in test_idx]

print(f'train/val/test = {len(train_paths)}/{len(val_paths)}/{len(test_paths)}')
print('train:', Counter(TARGET_COLORS[y] for y in train_labels))
print('val:  ', Counter(TARGET_COLORS[y] for y in val_labels))
print('test: ', Counter(TARGET_COLORS[y] for y in test_labels))

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

In [ ]:
class CarColorDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.labels[i]


loaders = {
    'train': DataLoader(CarColorDataset(train_paths, train_labels, train_tfms),
                        batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True),
    'val':   DataLoader(CarColorDataset(val_paths,   val_labels,   eval_tfms),
                        batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True),
    'test':  DataLoader(CarColorDataset(test_paths,  test_labels,  eval_tfms),
                        batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True),
}

In [ ]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    y_true, y_pred = [], []

    for xb, yb in tqdm(loader, desc='train', leave=False):
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        loss_val = loss.item() * xb.size(0)
        running_loss += loss_val
        preds = logits.argmax(dim=1)
        correct += (preds == yb).sum().item()
        total += xb.size(0)
        y_true.extend(yb.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    return running_loss / total, correct / total, f1


@torch.inference_mode()
def eval_epoch(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    y_true, y_pred = [], []

    for xb, yb in tqdm(loader, desc='eval', leave=False):
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = model(xb)
        loss = criterion(logits, yb)

        running_loss += loss.item() * xb.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == yb).sum().item()
        total += xb.size(0)
        y_true.extend(yb.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    return running_loss / total, correct / total, f1


def fit(model, loaders, epochs, criterion, optimizer, scheduler=None, patience=10, name='model'):
    history = {'train_loss': [], 'val_loss': [], 'train_f1': [], 'val_f1': []}
    best_val_f1, best_epoch, bad = -1.0, 0, 0

    for epoch in range(1, epochs + 1):
        t0 = time.time()

        train_loss, train_acc, train_f1 = train_epoch(model, loaders['train'], criterion, optimizer)
        val_loss, val_acc, val_f1 = eval_epoch(model, loaders['val'], criterion)

        if scheduler:
            scheduler.step(val_loss)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_f1'].append(train_f1)
        history['val_f1'].append(val_f1)

        print(f'[{name}] epoch {epoch:3d}  '
              f'train_loss={train_loss:.4f}  train_f1={train_f1:.4f}  '
              f'val_loss={val_loss:.4f}  val_f1={val_f1:.4f}  '
              f'time={time.time()-t0:.1f}s')

        if val_f1 > best_val_f1 + 1e-4:
            best_val_f1, best_epoch, bad = val_f1, epoch, 0
            torch.save(model.state_dict(), f'best_{name}.pt')
        else:
            bad += 1
            if bad >= patience:
                print(f'  Early stop (best val_f1={best_val_f1:.4f} @ epoch {best_epoch})')
                break

    print(f'[{name}] Done. Best val_f1 = {best_val_f1:.4f} @ epoch {best_epoch}')
    model.load_state_dict(torch.load(f'best_{name}.pt', map_location=DEVICE, weights_only=True))
    return model, history


def plot_history(history, name):
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].plot(history['train_loss'], label='train'); ax[0].plot(history['val_loss'], label='val')
    ax[0].set_title(f'{name} — Loss'); ax[0].legend(); ax[0].set_xlabel('epoch')
    ax[1].plot(history['train_f1'], label='train'); ax[1].plot(history['val_f1'], label='val')
    ax[1].set_title(f'{name} — F1_macro'); ax[1].legend(); ax[1].set_xlabel('epoch')
    plt.tight_layout(); plt.show()

In [ ]:
def conv3x3(in_ch, out_ch, stride=1):
    return nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = conv3x3(in_ch, out_ch, stride)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.relu  = nn.ReLU(inplace=True)
        self.conv2 = conv3x3(out_ch, out_ch)
        self.bn2   = nn.BatchNorm2d(out_ch)

        self.downsample = None
        if stride != 1 or in_ch != out_ch:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        return self.relu(out + identity)


class SmallResNet(nn.Module):
    def __init__(self, block, layers, num_classes=8):
        super().__init__()
        self.in_ch = 64

        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1),
        )

        self.layer1 = self._make_layer(block, 64,  layers[0], stride=1)
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Linear(512 * block.expansion, num_classes)

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01); nn.init.zeros_(m.bias)

    def _make_layer(self, block, planes, n_blocks, stride):
        layers = [block(self.in_ch, planes, stride)]
        self.in_ch = planes * block.expansion
        for _ in range(1, n_blocks):
            layers.append(block(self.in_ch, planes))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x).flatten(1)
        return self.fc(x)

In [ ]:
model_scratch = SmallResNet(BasicBlock, [2, 2, 2, 2], num_classes=NUM_CLASSES).to(DEVICE)
print(f'Params: {sum(p.numel() for p in model_scratch.parameters())/1e6:.2f}M')

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model_scratch.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5, min_lr=1e-6)

model_scratch, hist_scratch = fit(
    model_scratch, loaders, epochs=50,
    criterion=criterion, optimizer=optimizer, scheduler=scheduler,
    patience=10, name='ResNet-scratch'
)
plot_history(hist_scratch, 'ResNet-scratch')

In [ ]:
test_loss, test_acc, test_f1 = eval_epoch(model_scratch, loaders['test'], nn.CrossEntropyLoss())
print(f'ResNet-scratch  test_acc={test_acc:.4f}  test_f1_macro={test_f1:.4f}')

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

model_rn18 = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

in_features = model_rn18.fc.in_features
model_rn18.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_features, NUM_CLASSES),
)
model_rn18 = model_rn18.to(DEVICE)
print(f'ResNet18 params: {sum(p.numel() for p in model_rn18.parameters())/1e6:.2f}M')

In [ ]:
for p in model_rn18.parameters():
    p.requires_grad = False
for p in model_rn18.fc.parameters():
    p.requires_grad = True

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer_rn = optim.AdamW(model_rn18.fc.parameters(), lr=1e-3, weight_decay=1e-4)

model_rn18, hist_rn1 = fit(
    model_rn18, loaders, epochs=10,
    criterion=criterion, optimizer=optimizer_rn,
    patience=5, name='ResNet18-head'
)

In [ ]:
for name, p in model_rn18.named_parameters():
    p.requires_grad = (name.startswith('layer3') or
                       name.startswith('layer4') or
                       name.startswith('fc'))

optimizer_rn2 = optim.AdamW([
    {'params': model_rn18.layer3.parameters(), 'lr': 3e-5},
    {'params': model_rn18.layer4.parameters(), 'lr': 1e-4},
    {'params': model_rn18.fc.parameters(),     'lr': 5e-4},
], weight_decay=1e-4)

model_rn18, hist_rn2 = fit(
    model_rn18, loaders, epochs=30,
    criterion=criterion, optimizer=optimizer_rn2,
    patience=8, name='ResNet18-ft'
)
plot_history(hist_rn2, 'ResNet18 (fine-tune)')

In [ ]:
test_loss_rn, test_acc_rn, test_f1_rn = eval_epoch(model_rn18, loaders['test'], nn.CrossEntropyLoss())
print(f'ResNet18  test_acc={test_acc_rn:.4f}  test_f1_macro={test_f1_rn:.4f}')

In [ ]:
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

model_eff = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)

in_features = model_eff.classifier[1].in_features
model_eff.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_features, NUM_CLASSES),
)
model_eff = model_eff.to(DEVICE)
print(f'EfficientNet-B0 params: {sum(p.numel() for p in model_eff.parameters())/1e6:.2f}M')

In [ ]:
for p in model_eff.parameters():
    p.requires_grad = False
for p in model_eff.classifier.parameters():
    p.requires_grad = True

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer_eff = optim.AdamW(model_eff.classifier.parameters(), lr=1e-3, weight_decay=1e-4)

model_eff, hist_eff1 = fit(
    model_eff, loaders, epochs=10,
    criterion=criterion, optimizer=optimizer_eff,
    patience=5, name='EffNet-head'
)

In [ ]:
for i, stage in enumerate(model_eff.features):
    for p in stage.parameters():
        p.requires_grad = (i >= 6)

backbone_params = [p for n, p in model_eff.named_parameters()
                   if p.requires_grad and not n.startswith('classifier')]
head_params = [p for p in model_eff.classifier.parameters() if p.requires_grad]

optimizer_eff2 = optim.AdamW([
    {'params': backbone_params, 'lr': 3e-5},
    {'params': head_params,     'lr': 3e-4},
], weight_decay=1e-4)

model_eff, hist_eff2 = fit(
    model_eff, loaders, epochs=30,
    criterion=criterion, optimizer=optimizer_eff2,
    patience=8, name='EffNet-ft'
)
plot_history(hist_eff2, 'EfficientNet-B0 (fine-tune)')

In [ ]:
test_loss_eff, test_acc_eff, test_f1_eff = eval_epoch(model_eff, loaders['test'], nn.CrossEntropyLoss())
print(f'EfficientNet-B0  test_acc={test_acc_eff:.4f}  test_f1_macro={test_f1_eff:.4f}')

In [ ]:
import pandas as pd

def count_params(m):
    return sum(p.numel() for p in m.parameters()) / 1e6

df = pd.DataFrame([
    ('ResNet (scratch)',     count_params(model_scratch), test_f1),
    ('ResNet18 (pretrained FT)', count_params(model_rn18),    test_f1_rn),
    ('EfficientNet-B0 (FT)',    count_params(model_eff),     test_f1_eff),
], columns=['Model', 'Params (M)', 'Test F1_macro'])
df

## Выводы

1. **Своя ResNet с нуля** — базово работает, но без ImageNet-претрейна качество низкое.
2. **ResNet18 с ImageNet** — двухстадийный transfer learning (голова и разморозка верхних слоёв) даёт хороший результат.
3. **EfficientNet-B0** — компактнее по числу параметров, качество сравнимо с ResNet18.